# Regression Modeling

## Overview

In this section we're going to try various approaches to try and prove our whether or not we can predict a neighborhoods noise level based on various socioeconomic factors.

$H_0$: There is no association between a ZCTA's socioeconomic disadvantage and its average transportation noise exposure ($\beta_1 = 0$), controlling for geographic and demographic factors.

$H_1$: ZCTAs with higher levels of socioeconomic disadvantage (lower median household income, higher poverty rates, or higher disadvantage index scores) experience significantly higher mean transportation noise levels ($\beta_1 > 0$ or $\beta_1 \neq 0$).

### Pre-modeling observations

The most important pre-modeling observation is that decibels (dB) which we're trying to predict is not a linear variable, it actually grows exponentially like the earthquake magnitude scale. 30 decibels is 10 times louder than 20 decibles. 

Another important pre-modeling observation that we already established in the EDA section is that there is some pretty strong correlation between similar variables like income, poverty_rate, etc. 

Imports:

In [51]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.stattools import acf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import numpy as np

In [52]:
master = pd.read_csv("../data/processed/master_dataset.csv").dropna()
master.shape

(1407, 17)

## Original simple naive model


In [53]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

cleaned = master.dropna(subset=['noise_mean_db'])

train_df, test_df = train_test_split(cleaned, test_size=0.2)


formula = f"noise_mean_db ~ 1"
model = smf.ols(formula=formula, data=train_df).fit()

print(model.summary())


y_pred = model.predict(test_df)
y_test = test_df['noise_mean_db']

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
{'mea': mae, 'rmse': rmse, 'r2': r2}

                            OLS Regression Results                            
Dep. Variable:          noise_mean_db   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                       nan
Date:                Sun, 07 Jun 2026   Prob (F-statistic):                nan
Time:                        01:21:00   Log-Likelihood:                -2214.9
No. Observations:                1125   AIC:                             4432.
Df Residuals:                    1124   BIC:                             4437.
Df Model:                           0                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     55.5701      0.052   1074.991      0.0

{'mea': 1.3175664323831113,
 'rmse': np.float64(1.7178274306341492),
 'r2': -0.0028215939103761745}

#### Naive Model Analysis

As is typical for a Naive Model, there's not much to say. There's no $R^2$ because it requires weights or F-stat since the F-stat literally is a measure of how different your model is from the mean.

We know our mean though! It's roughly about 55.6 db.

Now we know what our baseline is!

## Original simple multi-linear regression model

In [54]:
#noise_max and noise_min encode too similar info to noise_mean so we're taking it out
predictors = [col for col in master.columns if 
              col != 'noise_mean_db' and col != 'noise_max_db' and col != 'noise_min_db'] 

formula = f"noise_mean_db ~ {' + '.join(predictors)}"
original_model = smf.ols(formula, data=master).fit()
print(original_model.summary())

                            OLS Regression Results                            
Dep. Variable:          noise_mean_db   R-squared:                       0.107
Model:                            OLS   Adj. R-squared:                  0.098
Method:                 Least Squares   F-statistic:                     11.89
Date:                Sun, 07 Jun 2026   Prob (F-statistic):           1.85e-26
Time:                        01:21:00   Log-Likelihood:                -2688.1
No. Observations:                1407   AIC:                             5406.
Df Residuals:                    1392   BIC:                             5485.
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

#### Original Model Analysis:

This model is actually not half bad! Our $R^2$ value is pretty low, but it's the difference in $R^2$ and adj. $R^2$ that we really care about. The fact that there's not a big difference after adjusting means we have some good variables

The prob F-Stat is amazing, there's definitely some statistically significant analysis here. 

Some of our coefficients though are kinda weird. Likely because of the strong correlation between various variables and also because our sample size is not the biggest.

Lastly, we get a warning about our condition number being huge, this is telling us again that there exists strong multicollinearity among some of our variables. 

#### Mini Permutation Test

In [55]:
original_f_stat = original_model.fvalue

permuted_f_stats = []
n_resamples = 1000

df_perm = master.copy()

for i in range(n_resamples):

    df_perm['noise_mean_db'] = np.random.permutation(df_perm['noise_mean_db'].values)
    
    perm_model = smf.ols(formula= formula, data=df_perm).fit()
    permuted_f_stats.append(perm_model.fvalue)

p_value = np.sum(np.array(permuted_f_stats) >= original_f_stat) / n_resamples
{'f-stat': original_f_stat, 'p-value': p_value}

{'f-stat': np.float64(11.892028266908683), 'p-value': np.float64(0.0)}

#### Mini Permutation Analysis

Just for good measure we can perm test to make sure we actually have a statistically viable model. It's a bit redundant, but what the multivariate permutation test lets us do is we can definitively say that our noise_level_db is not randomly distributed and there is a clear link between noise level and the various other variables. 

### We're going to explore two options for bettering our basic log-linear model

    1. Choosing columns manually

    2. Choosing columns using recommendations from Lasso regression

The obvious place to start is just using our pre existing knowledge of the data to make a decision of what variables to keep. Variables that sound redundant, likely are. 

### Option 1: Manually Dropping Rows

In [56]:
predictors

['zcta',
 'median_household_income',
 'unemployment_rate',
 'population',
 'pnhwhite',
 'pnhblack',
 'phispanic',
 'pforeign_born',
 'punemployed',
 'affluence',
 'disadvantage',
 'median_family_income',
 'home_value',
 'poverty_rate']

#### VIF

In [57]:
var_matrix = master[predictors].copy()
var_matrix = sm.add_constant(var_matrix)

vif_data = pd.DataFrame()
vif_data["Variable"] = var_matrix.columns
vif_data["VIF"] = [variance_inflation_factor(var_matrix.values, i) for i in range(var_matrix.shape[1])]

print(vif_data[vif_data["Variable"] != "const"].reset_index(drop=True))

                   Variable        VIF
0                      zcta   1.645779
1   median_household_income   6.872078
2         unemployment_rate   1.813949
3                population   1.519717
4                  pnhwhite  10.209955
5                  pnhblack   1.907648
6                 phispanic   6.690327
7             pforeign_born   4.052176
8               punemployed   1.472697
9                 affluence  10.924274
10             disadvantage   5.490869
11     median_family_income  12.511987
12               home_value   3.146004
13             poverty_rate   3.404937


#### VIF Takeaways

Clearly there's some correlation issues happening here. The especially bad offendors are median_household_income, pnhwhite, phispanic, affluence, and median_family_income. It also makes sense to drop them as we'll see in the next section!

There's 3 big categories to our data:

1. Socio-economic stattus: median_household_income, affluence, median_household_income, home_value
2. Poverty: unemployment rate, punemployed, disadvantage, poverty_rate
3. Race: pnhwhite, pnhblack, phispanic, pforeigh_born

And then there's our other singular features like:

4. zcta
5. population

As a reminder, our current analysis focuses just on mean noise so we're ignoring max and min.

So what makes most sense here:

For socio-economic status we could choose either median_household_income or median_household_income income, let's go with household! home_value is probably useful enough to keep too!

For poverty, poverty_rate is the obvious choice. It encapsulates unemployment rate and is less abstract than disadvantage.

For race it's probably fine to just drop pnhwhite and keep the rest. pnhwhite is just redundant info.

We should also drop zcta since the numerical number doesn't mean anything, but we should definitely keep population.


So our fun new model looks like this:

In [58]:
new_predictors = ['median_household_income', 'home_value', 'poverty_rate', 'pnhblack', 
              'phispanic', 'pforeign_born', 'population']

formula = f"noise_mean_db ~ {' + '.join(new_predictors)}"
improved_model = smf.ols(formula, data=master).fit()
print(improved_model.summary())

                            OLS Regression Results                            
Dep. Variable:          noise_mean_db   R-squared:                       0.076
Model:                            OLS   Adj. R-squared:                  0.071
Method:                 Least Squares   F-statistic:                     16.45
Date:                Sun, 07 Jun 2026   Prob (F-statistic):           6.02e-21
Time:                        01:21:08   Log-Likelihood:                -2711.9
No. Observations:                1407   AIC:                             5440.
Df Residuals:                    1399   BIC:                             5482.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

#### Manual Model Analysis:

This model is a bit better in some ways! 

Our condition number is smaller but still too big. but 

Our $R^2$ went down quite a bit though.

Our F-Stat got better but not by a lot.

I think we can do a lot better actually using techniques we learned in class. 

### Option 2: Lasso Regression


In [59]:
all_needed_columns = ['noise_mean_db'] + predictors

y = master['noise_mean_db']
#standardize variables
df_scaled = master[predictors].copy()
for col in predictors:
    if col != 'zcta':
        df_scaled[col] = (df_scaled[col] - df_scaled[col].mean()) / df_scaled[col].std()

df_scaled['noise_mean_db'] = y

formula = f"noise_mean_db ~ {' + '.join(predictors)}"
model = smf.ols(formula, data=df_scaled)

alphas = np.linspace(0.001, 1.0, 100)

best_alpha = None
lowest_bic = float('inf')  
best_params = None

y_true = df_scaled['noise_mean_db']
X_matrix = model.exog 
n_samples = len(y_true)

#using BIC to find the best alpha
for a in alphas:
    test_lasso = model.fit_regularized(alpha=a, L1_wt=1.0)
    
    y_pred = np.dot(X_matrix, test_lasso.params)
    sse = np.sum((y_true - y_pred) ** 2)
    
    k_features = np.sum(test_lasso.params[1:] != 0)
    
    if sse > 0:
        bic = n_samples * np.log(sse / n_samples) + k_features * np.log(n_samples)
    else:
        bic = float('inf')
    
    #keep best alpha
    if bic < lowest_bic:
        lowest_bic = bic
        best_alpha = a
        best_params = test_lasso.params

print({'best_alpha': best_alpha})

final_coefs = pd.DataFrame({
    'Feature': ['Intercept'] + predictors,
    'Coefficient': best_params
})


print([final_coefs['Coefficient']])

{'best_alpha': np.float64(0.05145454545454546)}
[Intercept                  53.204694
zcta                        0.000025
median_household_income     0.000000
unemployment_rate           0.180408
population                  0.000000
pnhwhite                   -0.168996
pnhblack                    0.100716
phispanic                   0.000000
pforeign_born               0.165322
punemployed                 0.000000
affluence                   0.127686
disadvantage                0.000000
median_family_income        0.000000
home_value                  0.000000
poverty_rate                0.000000
Name: Coefficient, dtype: float64]


Yippee!! We've figured out which variables are worth keeping, we can create a new, hopefully improved linear regression model based on just the coefficients that didn't go to 0.


In [60]:
features = ['zcta', 'unemployment_rate', 'pnhwhite', 
            'pnhblack', 'pforeign_born', 'affluence']
formula = f"noise_mean_db ~ {' + '.join(features)}"
lasso_improved_model = smf.ols(formula, data=master).fit()
print(lasso_improved_model.summary())

                            OLS Regression Results                            
Dep. Variable:          noise_mean_db   R-squared:                       0.099
Model:                            OLS   Adj. R-squared:                  0.095
Method:                 Least Squares   F-statistic:                     25.63
Date:                Sun, 07 Jun 2026   Prob (F-statistic):           5.10e-29
Time:                        01:21:09   Log-Likelihood:                -2694.3
No. Observations:                1407   AIC:                             5403.
Df Residuals:                    1400   BIC:                             5439.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            63.1216      2.58

#### Lasso Model Analysis:

We really improved our model! I mean, is it a great model? Not really... but we definitely did a lot better than just manually selecting and have addressed some of the issues with our original model.

Our condition number got way smaller! It's still a bit too big and there is still likely multicollinearity in our predictors but at least that plays way less of a factor.

Our $R^2$ stayed the same and our adjusted $R^2$ actually went up! We literally improved the predictive power of our model.

And our F-Stat is huge which is a great sign for statistical significance of the model.

### Conclusion:

At the end of the day, our model was hampered by one big issue and that is data! More data is always a good thing, unless you don't have the space to store it or the memory to do analysis on it. Nevertheless, I think we did a pretty good job!

There is a clear correlation between noise leveles and various socio-economic factors! Richer neighborhoods are quiter while neighborhoods with high unemployment rates or high percentages of foreign born residents tend to be louder. 

We saw clear model improvements after trying Lasso regression, but we also saw that even just having baseline knowledge about the data you're working with and trying to manually select can make a difference too!

If we would do this project all over again, we'd start with making our sample size bigger, maybe expanding to neighborhoods across the US instead of only focusing on California. If we were really serious data scientists we may even try some solutions to filling our NaN rows with reasonable inferences using some ML algorithm. 